In [38]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error


import warnings

warnings.simplefilter("ignore", FutureWarning)

pd.set_option('display.max_columns', None)

In [39]:
cols = [
    'year', 'month', 'day', 'hour', 'minute', 'second', # A–F
    'glucose_level', # G
    'finger_stick', # H
    'basal', # I
    'bolus', # J
    'sleep', # K
    'work', # L
    'stressors', # M
    'hypo_event', # N
    'illness', # O
    'exercise', # P
    'basis_heart_rate', # Q
    'basis_gsr', # R
    'basis_skin_temperature', # S
    'basis_air_temperature',  # T
    'basis_step', # U
    'basis_sleep', # V
    'meal', # W
    'meal_type' # X
]

In [40]:

PATIENTS = [559, 563, 570, 575, 588, 591]
HORITZO = {30:6, 60:12}  # minuts : pas de 5 mins

In [41]:
# Funcio per obtenir els datasets
def load_data(id: int, train_or_test: str) -> pd.DataFrame:
    df=pd.read_csv(f'../data/{id}/{id}_{train_or_test}.csv', sep=';', header = None, names = cols)
    return df


df_559_train = load_data(559, 'train')
df_559_test = load_data(559, 'test')

In [42]:
def preprocess(df: pd.DataFrame) -> pd.DataFrame:
    prep = df.copy()

    prep['timestamp'] = pd.to_datetime(
        dict(year=df.year, month=df.month, day=df.day, hour=df.hour, minute=df.minute)
    )
    
    prep.sort_values('timestamp', inplace=True)

    # Coma decimal a punt
    convert = ["basal","bolus","basis_gsr","basis_skin_temperature","basis_air_temperature"]
    
    for c in convert:
        prep[c] = (prep[c].astype(str)
                   .str.replace(",",".", regex=False)
                   .str.strip()
                   .astype(float))

    # Unifiquem tipo
    cat_meal = {
        1:"Desayuno",
        2:"Almuerzo",
        3:"Cena",
        4:"Snack",
        5:"Correccion_hipo"
    }

    prep["meal_type"] = prep["meal_type"].map(cat_meal).astype("category")
    prep = pd.get_dummies(prep, columns=['meal_type'], dummy_na=False, prefix='meal')

    # Drop columnas amb casi tot NaN o valor constant
    prep = prep.drop(columns=["second","finger_stick","meal"])

    # Zeros que no poden ser valids
    invalid_zero = [
        "glucose_level",
        "basis_heart_rate",
        "basis_gsr",
        "basis_skin_temperature",
        "basis_air_temperature"
    ]
    
    prep[invalid_zero] = prep[invalid_zero].replace(0, np.nan)
    prep[invalid_zero] = prep[invalid_zero].fillna(method='ffill')

    prep = prep.dropna(subset=['glucose_level'])


    prep = prep.drop(columns=['timestamp'])
    return prep

In [43]:
df_559_train = preprocess(df_559_train)
df_559_test = preprocess(df_559_test)

display(df_559_train.tail())
display(df_559_test.head())
display(df_559_test.tail())

,year,month,day,hour,minute,glucose_level,basal,bolus,sleep,work,stressors,hypo_event,illness,exercise,basis_heart_rate,basis_gsr,basis_skin_temperature,basis_air_temperature,basis_step,basis_sleep,meal_Almuerzo,meal_Cena,meal_Correccion_hipo,meal_Desayuno,meal_Snack
12078,2022,1,17,23,35,161.0,0.83,0.0,3,0,0,0,0,0,58.0,0.000213,92.3,89.6,0,94,False,False,False,False,False
12079,2022,1,17,23,40,164.0,0.83,0.0,3,0,0,0,0,0,58.0,0.000201,92.3,89.6,0,94,False,False,False,False,False
12080,2022,1,17,23,45,168.0,0.83,0.0,3,0,0,0,0,0,58.0,0.000198,92.3,89.6,0,94,False,False,False,False,False
12081,2022,1,17,23,50,172.0,0.83,0.0,3,0,0,0,0,0,57.0,0.000192,92.3,89.6,0,94,False,False,False,False,False
12082,2022,1,17,23,55,176.0,0.83,0.0,3,0,0,0,0,0,58.0,0.000188,92.3,89.6,0,94,False,False,False,False,False


,year,month,day,hour,minute,glucose_level,basal,bolus,sleep,work,stressors,hypo_event,illness,exercise,basis_heart_rate,basis_gsr,basis_skin_temperature,basis_air_temperature,basis_step,basis_sleep,meal_Almuerzo,meal_Cena,meal_Desayuno,meal_Snack
0,2022,1,18,0,0,179.0,0.83,0.0,3,0,0,0,0,0,57.0,0.000183,92.3,89.6,0,94,False,False,False,False
1,2022,1,18,0,5,183.0,0.83,0.0,3,0,0,0,0,0,57.0,0.000182,92.3,89.6,0,94,False,False,False,False
2,2022,1,18,0,10,187.0,0.83,0.0,3,0,0,0,0,0,57.0,0.000177,92.3,89.6,0,94,False,False,False,False
3,2022,1,18,0,15,191.0,0.83,0.0,3,0,0,0,0,0,56.0,0.000169,92.3,89.6,0,94,False,False,False,False
4,2022,1,18,0,20,195.0,0.83,0.0,3,0,0,0,0,0,56.0,0.000166,92.3,89.6,0,94,False,False,False,False


,year,month,day,hour,minute,glucose_level,basal,bolus,sleep,work,stressors,hypo_event,illness,exercise,basis_heart_rate,basis_gsr,basis_skin_temperature,basis_air_temperature,basis_step,basis_sleep,meal_Almuerzo,meal_Cena,meal_Desayuno,meal_Snack
2871,2022,1,27,23,15,185.0,1.25,0.0,3,0,0,0,0,0,150.0,0.000071,80.96,79.7,0,0,False,False,False,False
2872,2022,1,27,23,20,183.0,1.25,0.0,3,0,0,0,0,0,150.0,0.000071,80.96,79.7,0,0,False,False,False,False
2873,2022,1,27,23,25,182.0,1.25,0.0,3,0,0,0,0,0,150.0,0.000071,80.96,79.7,0,0,False,False,False,False
2874,2022,1,27,23,30,180.0,1.25,0.0,3,0,0,0,0,0,150.0,0.000071,80.96,79.7,0,0,False,False,False,False
2875,2022,1,27,23,35,177.0,1.25,0.0,3,0,0,0,0,0,150.0,0.000071,80.96,79.7,0,0,False,False,False,False


In [44]:
print('Shape train: ', df_559_train.shape)
print('Shape test: ', df_559_test.shape)
print(df_559_train.isnull().mean()*100)

Shape train:  (12081, 25)
Shape test:  (2876, 24)
year                      0.000000
month                     0.000000
day                       0.000000
hour                      0.000000
minute                    0.000000
glucose_level             0.000000
basal                     0.000000
bolus                     0.000000
sleep                     0.000000
work                      0.000000
stressors                 0.000000
hypo_event                0.000000
illness                   0.000000
exercise                  0.000000
basis_heart_rate          1.158844
basis_gsr                 1.158844
basis_skin_temperature    1.158844
basis_air_temperature     1.158844
basis_step                0.000000
basis_sleep               0.000000
meal_Almuerzo             0.000000
meal_Cena                 0.000000
meal_Correccion_hipo      0.000000
meal_Desayuno             0.000000
meal_Snack                0.000000
dtype: float64


In [45]:
def make_xy(df: pd.DataFrame, steps: int):

    y = df['glucose_level'].shift(-steps)
    X = df.drop(columns=['glucose_level'])

    mask = y.notna()

    return X[mask], y[mask]

def evaluate(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    return rmse, mae

In [46]:
# ------------------------------------------------
# Entrenamiento, predicción y métrica paciente por paciente
# ------------------------------------------------
all_metrics = []

for id in PATIENTS:
    print(f'\nPaciente {id}')
    train_raw = load_data(id, 'train')
    test_raw  = load_data(id, 'test')

    train = preprocess(train_raw)
    test  = preprocess(test_raw)

    
    # igualamos columnas de train y test (outer join) y rellenamos lo que falte con 0
    train, test = train.align(test, join='outer', axis=1, fill_value=0)

    # ignorar los primeros 60 min del test
    test = test.iloc[12:].reset_index(drop=True)

    for minuts, steps in HORITZO.items():
        # ---------- entrenamiento offline ----------
        X_train, y_train = make_xy(train, steps)

        model = RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=-1
        )
        model.fit(X_train, y_train)

        # ---------- predicción en test ----------
        X_test = test.iloc[:-steps].drop(columns=['glucose_level'])

        y_true = test['glucose_level'].iloc[:-steps].reset_index(drop=True)
        y_pred = model.predict(X_test)

        # guardar CSV requerido
        out_csv = f'../data/predicted/pred{id}_{minuts}min.csv'

        df_pred = pd.DataFrame({'idx_original': X_test.index, f'pred_glucose_t+{minuts}': y_pred})
        df_pred.to_csv(out_csv, index=False)

        # ---------- métricas ----------
        rmse, mae = evaluate(y_true, y_pred)
        all_metrics.append({
            'Paciente': id,
            'Horizonte': f'{minuts} min',
            'RMSE': rmse,
            'MAE' : mae
        })
        print(f'{minuts} min: RMSE={rmse:.2f}  MAE={mae:.2f}')




Paciente 559
30 min: RMSE=63.10  MAE=48.48
60 min: RMSE=66.75  MAE=51.92

Paciente 563
30 min: RMSE=53.81  MAE=41.46
60 min: RMSE=55.01  MAE=43.12

Paciente 570
30 min: RMSE=81.23  MAE=67.93
60 min: RMSE=81.38  MAE=68.00

Paciente 575
30 min: RMSE=94.14  MAE=80.19
60 min: RMSE=94.58  MAE=76.50

Paciente 588
30 min: RMSE=52.72  MAE=40.84
60 min: RMSE=53.03  MAE=41.26

Paciente 591
30 min: RMSE=63.61  MAE=51.80
60 min: RMSE=64.19  MAE=52.41


In [47]:
# ------------------------------------------------
# Tabla final (promedio incluido)
# ------------------------------------------------
metrics_df = pd.DataFrame(all_metrics)
prom = metrics_df.groupby('Horizonte')[['RMSE','MAE']].mean().reset_index()
prom.insert(0, 'Paciente', 'PROMEDIO')
result = pd.concat([metrics_df, prom], ignore_index=True)
print('\n==== RESULTADOS FINALES ====')
print(result.to_string(index=False))


==== RESULTADOS FINALES ====
Paciente Horizonte      RMSE       MAE
     559    30 min 63.097312 48.480239
     559    60 min 66.745377 51.921168
     563    30 min 53.807919 41.460131
     563    60 min 55.011178 43.120355
     570    30 min 81.234566 67.931094
     570    60 min 81.377533 67.998999
     575    30 min 94.136141 80.192753
     575    60 min 94.577509 76.503031
     588    30 min 52.716952 40.844879
     588    60 min 53.029047 41.262178
     591    30 min 63.613806 51.798395
     591    60 min 64.191970 52.408114
PROMEDIO    30 min 68.101116 55.117915
PROMEDIO    60 min 69.155436 55.535641
